In [10]:
# ADDING THE DATA

import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/credit_default_features.csv")

df.head()
print(df.shape)

(30000, 43)


In [11]:
# Build X (features) and y (target)

target = "default"

drop_cols = [
    "id",
    target,                       # avoid leaking target into X
    "bill_amt1","bill_amt2","bill_amt3",
    "bill_amt4","bill_amt5","bill_amt6",
    "pay_amt1","pay_amt2","pay_amt3",
    "pay_amt4","pay_amt5","pay_amt6",
]

X = df.drop(columns=drop_cols)
y = df[target]

print(f"X: {X.shape}, y: {y.shape}, default rate: {y.mean():.4f}")

X: (30000, 29), y: (30000,), default rate: 0.2212


In [12]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,                   # preserves 22% default rate in both splits
    random_state=RANDOM_STATE,
)

print(f"Train: {X_train.shape} ({y_train.mean():.4f})")
print(f"Test:  {X_test.shape} ({y_test.mean():.4f})")

Train: (24000, 29) (0.2212)
Test:  (6000, 29) (0.2212)


In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on train ONLY
X_test_scaled  = scaler.transform(X_test)        # transform with train stats

print(f"mean ~0: {X_train_scaled.mean():.4f}, std ~1: {X_train_scaled.std():.4f}")

mean ~0: -0.0000, std ~1: 1.0000


In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, classification_report

lr = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",      # upweights minority class (defaults)
    random_state=RANDOM_STATE,
)
lr.fit(X_train_scaled, y_train)

lr_proba = lr.predict_proba(X_test_scaled)[:, 1]
lr_pred  = (lr_proba >= 0.5).astype(int)

print("=== Logistic Regression ===")
print(f"ROC-AUC: {roc_auc_score(y_test, lr_proba):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, lr_proba):.4f}")
print(f"F1:      {f1_score(y_test, lr_pred):.4f}")
print(classification_report(y_test, lr_pred, digits=4))

=== Logistic Regression ===
ROC-AUC: 0.7542
PR-AUC:  0.5041
F1:      0.5082
              precision    recall  f1-score   support

           0     0.8726    0.7884    0.8283      4673
           1     0.4438    0.5946    0.5082      1327

    accuracy                         0.7455      6000
   macro avg     0.6582    0.6915    0.6683      6000
weighted avg     0.7777    0.7455    0.7575      6000



In [16]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,                 # cap depth to prevent overfitting
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,                    # use all CPU cores
)
rf.fit(X_train, y_train)          # trees don't need scaled data

rf_proba = rf.predict_proba(X_test)[:, 1]
rf_pred  = (rf_proba >= 0.5).astype(int)

print("=== Random Forest ===")
print(f"ROC-AUC: {roc_auc_score(y_test, rf_proba):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, rf_proba):.4f}")
print(f"F1:      {f1_score(y_test, rf_pred):.4f}")
print(classification_report(y_test, rf_pred, digits=4))

=== Random Forest ===
ROC-AUC: 0.7753
PR-AUC:  0.5586
F1:      0.5391
              precision    recall  f1-score   support

           0     0.8804    0.8143    0.8460      4673
           1     0.4827    0.6104    0.5391      1327

    accuracy                         0.7692      6000
   macro avg     0.6815    0.7123    0.6926      6000
weighted avg     0.7924    0.7692    0.7781      6000



In [17]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=4,                  # shallow trees, sequentially corrected
    learning_rate=0.05,
    random_state=RANDOM_STATE,
)
gb.fit(X_train, y_train)

gb_proba = gb.predict_proba(X_test)[:, 1]
gb_pred  = (gb_proba >= 0.5).astype(int)

print("=== Gradient Boosting ===")
print(f"ROC-AUC: {roc_auc_score(y_test, gb_proba):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, gb_proba):.4f}")
print(f"F1:      {f1_score(y_test, gb_pred):.4f}")
print(classification_report(y_test, gb_pred, digits=4))

=== Gradient Boosting ===
ROC-AUC: 0.7790
PR-AUC:  0.5579
F1:      0.4658
              precision    recall  f1-score   support

           0     0.8389    0.9484    0.8903      4673
           1     0.6639    0.3587    0.4658      1327

    accuracy                         0.8180      6000
   macro avg     0.7514    0.6536    0.6780      6000
weighted avg     0.8002    0.8180    0.7964      6000



In [18]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "Gradient Boosting"],
    "ROC-AUC": [
        roc_auc_score(y_test, lr_proba),
        roc_auc_score(y_test, rf_proba),
        roc_auc_score(y_test, gb_proba),
    ],
    "PR-AUC": [
        average_precision_score(y_test, lr_proba),
        average_precision_score(y_test, rf_proba),
        average_precision_score(y_test, gb_proba),
    ],
    "F1 (thr=0.5)": [
        f1_score(y_test, lr_pred),
        f1_score(y_test, rf_pred),
        f1_score(y_test, gb_pred),
    ],
})

print(results.to_string(index=False))

              Model  ROC-AUC   PR-AUC  F1 (thr=0.5)
Logistic Regression 0.754197 0.504050      0.508213
      Random Forest 0.775299 0.558624      0.539101
  Gradient Boosting 0.779021 0.557881      0.465753
